# Phase 3b -- K-stress addendum (A100-40GB, High-RAM **OFF**)

Phase 2's K marginal was ~flat (0.94-1.03x) -- a real null for its regime: KV demand
peaked at ~18% of the pool, so K's capacity channel never engaged. This notebook creates
the pressure deliberately and runs on the **40GB** A100 (High-RAM OFF -- the toggle pins
the variant, PREREQ_RESULTS Check 1), because on 40GB **both** ceilings fit inside a
small grid:

- pool ~ 0.85x40GB - ~16GB weights - overhead ~ **~120-135k FP16-KV tokens**
- per-request context ~ 7.4k-token unique document + 256 output ~ 7.7k tokens
- -> **FP16-KV ceiling ~ 16 concurrent requests; FP8-KV ~ 32**

Base grid: {FP16-KV, FP8-KV} x concurrency **{8, 16, 32, 48}** x 2 repeats = 16 cells,
2 server launches. Each level has a job: 8 = below both ceilings (the Phase-2 null must
reproduce), 16 = ON the FP16 knee, 32 = FP16 saturated / ON the FP8 knee (max
divergence), 48 = beyond both -- FP8's own plateau at ~2x FP16's, the demonstration the
80GB design could never fit in a sane concurrency range.

**With W-corners (now generated, `configs/k_stress/` has 32 configs, 4 server groups):**
AWQ frees ~10.4GB of weight memory (16.06 -> 5.66 GB), raising the FP16-KV ceiling to
~26 and the FP8-KV ceiling to ~52 -- still inside the same {8,16,32,48} grid. This tests
whether W has its own capacity-relief channel independent of K's precision, not just its
already-measured compute/bandwidth channel.

**Budget:** base grid ~1.5-2h (~20 units, worst-case). With W-corners: ~3-4h total
(~40 units), 4 server launches instead of 2. Independent of Sessions A-C -- run it on
either account, any time.

---
**Re-run note (2026-07-10):** the first attempt returned 16/16 `status: partial` with a
uniform ~9-11% request error rate -- vLLM `400 Bad Request` on documents whose real token
count overshot `max_model_len - max_new_tokens` (word-count sizing underestimated NQ
tokenization). Root cause fixed: documents are now sized **tokenizer-exactly** and every
prompt is verified against a hard `prompt_token_budget: 7900` at build time (a violation
fails the build loudly instead of 400ing mid-run). Re-running this notebook re-executes
all cells (partials are not skipped by resume). Expect `prompt_tokens_mean` ~ 7,460-7,500
in the records -- comfortably under 7,936.


---
**Session scope (2026-07-10):** this notebook now runs THREE corner sets in one session
(40 cells, 6 server launches, ~35–45 units):
1. **K-isolation** (fp16kv/fp8kv × conc {8,16,32,48} × 2 reps) — the capacity experiment.
2. **W capacity corners** (AWQ × both KV dtypes, same grid) — does freeing ~10.4 GB of
   weight memory raise the sustainable-concurrency ceiling (~16 → ~26 predicted)?
3. **KS long-context probe** (EAGLE-3 × both KV dtypes × conc {1, 8} × 2 reps) — the
   K-under-S contrast at 7.4k-token context, same hardware/kernels as the factorial's
   short-context KS (~×0.63 at c1). Deliberately below the capacity ceiling so the
   economics comparison stays clean. Together with the conc-8 K-isolation cells (the
   long-context K-solo measurement, free) this completes the 2×2 grid that decomposes
   emulation tax vs bandwidth credit without an H100.

---
**Bug A/B hardening (2026-07-11):** delete any `configs/k_stress/_parked/` directory from
the previous session — the probe configs are back in the main set and the sweep now
orders them last. Harness fixes: teardown signals the whole process group (vLLM V1's
EngineCore child can no longer be orphaned holding ~16GB), launch refuses to start on an
occupied GPU (one precise error instead of a cascade), a stalled launch fails in ~10 min
instead of 40, and each record now captures the selected attention backend from the
server log (`env.attention_backend`). If the eagle3-fp16kv launch stalls again, see the
"Rung 2" comment in its config files.

In [ ]:
# 1. Verify the 40GB card (High-RAM OFF)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import subprocess
gpu = subprocess.run(["nvidia-smi","--query-gpu=name","--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
assert "A100" in gpu, "Not an A100 - reconnect until Colab honors the selection."
mem = int(subprocess.run(["nvidia-smi","--query-gpu=memory.total","--format=csv,noheader,nounits"],
                         capture_output=True, text=True).stdout.strip().splitlines()[0])
assert mem < 50000, ("Got the 80GB A100 (%d MiB). Toggle High-RAM OFF and reconnect: "
                     "on 80GB the FP16 ceiling (~47) escapes this notebook grid.") % mem
units_before = input("Compute-units balance right now: ")

In [ ]:
# 2. Repo + Spec-Bench (long documents are built from its RAG passages)
%cd /content
!git clone https://github.com/manojarulmurugan/SpecDecoding-Study-vLLM-SGLang.git repo 2>/dev/null || (cd repo && git pull)
%cd /content/repo
!git clone --depth 1 https://github.com/hemingkx/Spec-Bench.git external/Spec-Bench 2>/dev/null || echo "Spec-Bench already present"
!test -f external/Spec-Bench/data/spec_bench/question.jsonl && echo "RAG data OK"

In [ ]:
# 3. Isolated vLLM environment (PREREQ_RESULTS Check 6 recipe; ~6-8 min)
%pip install -q virtualenv
!virtualenv -q /content/vllm_env
!/content/vllm_env/bin/pip install -q vllm==0.24.0 datasets pyyaml requests pytest ninja
!/content/vllm_env/bin/python -c "import vllm; print('vllm', vllm.__version__)"

In [ ]:
# 4. HF auth
import os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
print("token set:", bool(os.environ["HF_TOKEN"]))

In [ ]:
# 4b. Pre-download every checkpoint ONCE, before any server launch --
# bounded, retried, loud. scripts/predownload.py wraps `hf download` with a
# hard per-attempt timeout (30 min), 3 attempts with backoff, and a clear
# failure message: a bare `hf download` can sit silent indefinitely when
# HF's CDN misbehaves (2026-07-14: hub-issued xet-bridge URLs failed with
# "403 SignatureError: invalid key pair id" across unrelated repos --
# HF-side infra; `pip uninstall hf-xet` did NOT help; if this cell fails
# with that signature, wait a few hours and re-run this cell only).
# Re-runs are no-ops on a warm cache. Debug transcript:
# colab/archive_phase3b_xet_debug_20260714.ipynb
!/content/vllm_env/bin/python scripts/predownload.py

In [ ]:
# 5. Harness self-check (~1 min, no GPU)
!PATH="/content/vllm_env/bin:$PATH" /content/vllm_env/bin/python -m pytest tests -q

In [ ]:
# 6. Sanity: the two server commands (identical except --kv-cache-dtype fp8)
!PATH="/content/vllm_env/bin:$PATH" /content/vllm_env/bin/python -m harness.sweep "configs/k_stress/kstress_*.yaml" --results-dir results --dry-run 2>&1 | grep "server command"

In [ ]:
# 7. THE SWEEP: 40 cells, 6 launches, resumable. Cells at conc 32/48 under
# FP16-KV are SLOW BY DESIGN (preemption thrash) -- that's the phenomenon,
# not a hang; watch results/server_logs/ if unsure.
!PATH="/content/vllm_env/bin:$PATH" /content/vllm_env/bin/python -m harness.sweep "configs/k_stress/kstress_fp16kv_*.yaml" "configs/k_stress/kstress_fp8kv_*.yaml" "configs/k_stress/kstress_w4a16-*.yaml" "configs/k_stress/kstress_eagle3-*.yaml" --results-dir results

In [ ]:
# 8. Pool size from the FP16-KV server log -> predicted-vs-measured plateau
!grep -h "GPU KV cache size" results/server_logs/*.log | tail -2
POOL = input("FP16-KV pool size in tokens (from the FP16 group line above): ").replace(",", "")
!PATH="/content/vllm_env/bin:$PATH" /content/vllm_env/bin/python -m analysis.k_stress results --pool-tokens $POOL

In [ ]:
# 9. Preserve everything
units_after = input("Compute-units balance now: ")
print("Units burned:", units_before, "->", units_after)
!zip -qr phase3b_kstress_results.zip results
from google.colab import files
files.download("phase3b_kstress_results.zip")

## Reading the result

The four-row table should tell a two-knee story:

- **conc 8:** ratio ≈ 0.95–1.05, kv-usage well below 1.0, zero preemptions on both —
  the Phase-2 null reproducing below the ceiling. This row is what makes the rest
  credible.
- **conc 16:** FP16 kv-usage max ≈ 1.0 for the first time (batch pinned at the pool
  limit), FP8 cruising at ~0.5. Goodput ratio starts moving.
- **conc 32:** maximum divergence — FP16 batch flat at ~16 with preemptions > 0 and TTFT
  p95 blowing up; FP8 batch ~32, hitting ITS knee (kv-usage → 1.0).
- **conc 48:** both plateaued, FP8 plateau ≈ 2× FP16 — "FP8-KV doubles sustainable
  concurrency" measured on both sides, not extrapolated.

Cross-check the predicted plateau line (pool ÷ measured mean context) against the
measured batch columns; agreement within ~10% closes the loop between the capacity
arithmetic and the observed behavior. Decision-guide sentence this feeds: *enable FP8-KV
when projected demand (concurrency × context × 128 KiB) approaches the pool; below that
it costs ~5% on A100 (emulation tax) and buys nothing.*

**W capacity section:** the AWQ table's FP16-KV column should plateau near ~26 (vs ~16
for FP16 weights) — weight quantization as a KV-capacity lever, measured.

**KS probe section:** the decisive column is the K-under-S ratio vs the K-solo ratio at
the same concurrency, both at long context. If K-under-S climbs from the factorial's
~×0.63 toward the K-solo value, context length buys back bandwidth credit and QuantSpec's
regime-dependence is confirmed; if it stays near ×0.63 while τ stays flat, the emulation
tax dominates at any context length on A100. Either outcome is a clean sentence for the
decision guide.